# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [ ]:
print('Ready. No external packages required for this practical.')

## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [ ]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [3]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
None


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [ ]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [ ]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [2]:
# 1. Base Helper Functions (from Step 2 & 3)
def match(pattern, tokens):
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith("?"):
            bindings[p] = tokens[i:]  # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None  # constant word did not match
    return bindings if len(pattern) == len(tokens) else None


SWAP = {"my": "your", "i": "you", "me": "you", "am": "are", "mine": "yours"}


def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]


# 2. Task 1 Extended RULES table
RULES = [
    # Original rules
    (["i", "am", "?X"], "How long have you been {X}?"),
    (["i", "need", "?X"], "Why do you need {X}?"),
    (["i", "cannot", "?X"], "What stops you being able to {X}?"),
    # 5 New rules
    (["i", "feel", "?X"], "What makes you feel {X}?"),
    (["i", "want", "?X"], "Why is {X} important to you?"),
    (["hello"], "Hello! How can I help you today?"),
    (["why", "?X"], "Why do you think {X}?"),
    (["my", "favorite", "?X"], "Tell me more about why you like {X}."),
]

FALLBACK = "Tell me more about that."


# 3. Responder Function
def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            if "?X" in b:
                captured = swap_person(b["?X"])
                return template.format(X=" ".join(captured))
            return template
    return FALLBACK


# 4. Test all 5 new rules
test_sentences = [
    "I feel very tired today",
    "I want a new laptop",
    "Hello",
    "Why is the sky blue",
    "My favorite food is pizza",
]

for s in test_sentences:
    print(f"{s:<30} -> {respond(s)}")

I feel very tired today        -> What makes you feel very tired today?
I want a new laptop            -> Why is a new laptop important to you?
Hello                          -> Hello! How can I help you today?
Why is the sky blue            -> Why do you think is the sky blue?
My favorite food is pizza      -> Tell me more about why you like food is pizza.


### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [4]:
def match_advanced(pattern, tokens):
    """Matches patterns with a variable anywhere (start, middle, or end).

    Returns a dict of bindings, or None if there is no match.
    """
    # Find variable token and its index
    var_name = None
    var_idx = -1
    for i, p in enumerate(pattern):
        if p.startswith("?"):
            var_name = p
            var_idx = i
            break

    # If no variable is present, check exact token match
    if var_name is None:
        return {} if pattern == tokens else None

    # Prefix before variable and suffix after variable
    prefix = pattern[:var_idx]
    suffix = pattern[var_idx + 1 :]

    # Check if prefix matches start of tokens
    if tokens[: len(prefix)] != prefix:
        return None

    # Check if suffix matches end of tokens
    if suffix:
        if tokens[-len(suffix) :] != suffix:
            return None
        captured = tokens[len(prefix) : -len(suffix)]
    else:
        captured = tokens[len(prefix) :]

    # Ensure captured is not empty
    if len(captured) == 0:
        return None

    return {var_name: captured}


# Tests showing middle variable matching
pattern = ["i", "?X", "about", "exams"]

print(match_advanced(pattern, "i am worried about exams".split()))
print(match_advanced(pattern, "i feel very stressed about exams".split()))
print(match_advanced(pattern, "i happy about exams".split()))
print(match_advanced(pattern, "she is worried about exams".split()))  # None

{'?X': ['am', 'worried']}
{'?X': ['feel', 'very', 'stressed']}
{'?X': ['happy']}
None


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [5]:
def simplify(expr):
    """Recursively applies algebraic rewrite rules to nested lists.

    Rules applied:
    1. 1 * x  -> x
    2. x * 1  -> x
    3. 0 + x  -> x
    4. x + 0  -> x
    5. 0 * x  -> 0
    6. x * 0  -> 0
    """
    if not isinstance(expr, list):
        return expr

    # Simplify sub-expressions first
    left = simplify(expr[0])
    op = expr[1]
    right = simplify(expr[2])

    # Rule 1 & 2: 1 * x = x, x * 1 = x
    if op == "*":
        if left == 1:
            return right
        if right == 1:
            return left
        if left == 0 or right == 0:
            return 0

    # Rule 3 & 4: 0 + x = x, x + 0 = x
    if op == "+":
        if left == 0:
            return right
        if right == 0:
            return left

    return [left, op, right]


def simplify_until_fixed(expr):
    """Repeatedly simplifies until the expression stops changing."""
    while True:
        simplified = simplify(expr)
        if simplified == expr:
            return simplified
        expr = simplified


# Expression: 1 * x + 0 -> represented as [[1, '*', 'x'], '+', 0]
expression = [[1, "*", "x"], "+", 0]
result = simplify_until_fixed(expression)

print("Original Expression :", expression)
print("Simplified Result   :", result)

Original Expression : [[1, '*', 'x'], '+', 0]
Simplified Result   : x


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [ ]:
# TODO: no code needed - record your findings in the observations table

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

In [ ]:
# TODO: write your comparison in a markdown cell below this one

What the program actually stores:

1. A dictionary of word swaps (SWAP) mapping first-person words to second-person words.

2. A sequence of pattern templates paired with formatting string templates (RULES).

3. A default string response (FALLBACK).

What the user assumes the program stores:

1. An internal model of emotional and psychological concepts (e.g., "stress", "exams").

2. Knowledge about personal history, time, and human relationships.

3. An intention to hold a real conversation.

The Gap: The program processes symbols based purely on syntactic shape without any semantic understanding or context awareness.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| I feel very stressed today | ['i', 'feel', '?X'] | What makes you feel very stressed today? | Yes |
|I am worried about my exams  | ['i', 'am', '?X'] | How long have you been worried about your exams? | Yes |
|My favorite food is pizza  | ['my', 'favorite', '?X'] | Tell me more about why you like food is pizza. |Partially  |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| Assumed the program understood stress, emotions, and personal academic situations. | Stores only string patterns, predefined templates, and word replacement dictionaries. |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. It understands nothing. It operates purely on symbol manipulation and syntax matching (checking shapes of lists and strings) without any reference to real-world semantics or meaning.

2. Swapping "my" to "your" and "i" to "you" creates the illusion of active listening and empathetic perspective-taking, triggering human psychological projections of conversational engagement.

3. t highlights that human conversational evaluation is heavily subjective; humans tend to attribute intentionality and intelligence to systems based on surface-level linguistic fluency (Eliza Effect).

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [ ]:
NAME = "Fazeel Shahid"
ROLL_NUMBER = "2025-cse-072"
print(NAME, ROLL_NUMBER)

Ayesha Afzal 2025-cse-066
